<a href="https://colab.research.google.com/github/Renatops2910/Trabalho/blob/main/scraping_locafacil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re

In [ ]:
base_url = "https://computacao-aplicada.github.io/scraping/"

paginas = [
    base_url + "pagina1.html",
    base_url + "pagina2.html"
]

paginas

['https://computacao-aplicada.github.io/scraping/pagina1.html',
 'https://computacao-aplicada.github.io/scraping/pagina2.html']

In [ ]:
def limpar_texto(texto):
    if texto:
        texto = re.sub(r"\s+", " ", texto)
        return texto.strip()
    return None

In [ ]:
def extrair_anuncio(texto):

    codigo = re.search(r"LOC\d+", texto)
    codigo = codigo.group() if codigo else None

    preco = re.search(r"R\$\s*[\d\.,]+", texto)
    preco = preco.group() if preco else None

    quartos = re.search(r"Quartos\s+(\d+)", texto)
    quartos = quartos.group(1) if quartos else None

    banheiros = re.search(r"Banheiros\s+(\d+)", texto)
    banheiros = banheiros.group(1) if banheiros else None

    vagas = re.search(r"Vagas\s+(\d+)", texto)
    vagas = vagas.group(1) if vagas else None

    area = re.search(r"(\d+)\s*m²", texto)
    area = area.group(1) if area else None

    return {
        "codigo": codigo,
        "preco": preco,
        "area_m2": area,
        "quartos": quartos,
        "banheiros": banheiros,
        "vagas": vagas,
        "texto_completo": texto
    }

In [ ]:
dados = []

for url in paginas:

    resposta = requests.get(url)
    soup = BeautifulSoup(resposta.text, "html.parser")

    links = soup.find_all("a")

    for link in links:

        texto = limpar_texto(link.get_text(" ", strip=True))

        if texto and "Código: LOC" in texto:
            anuncio = extrair_anuncio(texto)
            dados.append(anuncio)

len(dados)

50

In [ ]:
df = pd.DataFrame(dados)

df.head()

,codigo,preco,area_m2,quartos,banheiros,vagas,texto_completo
0,LOC1001,"R$ 3.200,00",35,2,1,0,Sobrado aceita pet mobiliado Código: LOC1001 R...
1,LOC1002,"R$ 2.900,00",95,0,2,0,"Casa mobiliado Código: LOC1002 R$ 2.900,00 Cas..."
2,LOC1003,"R$ 0,00",110,2,1,0,Sobrado Código: LOC1003 Sobrado em rua tranqui...
3,LOC1004,"R$ 1.400,00",72,1,2,0,Apartamento aceita pet mobiliado Código: LOC10...
4,LOC1005,"R$ 2.900,00",68,1,2,1,"Studio mobiliado Código: LOC1005 R$ 2.900,00 S..."


In [ ]:
# remover anúncios sem código
df = df.dropna(subset=["codigo"])

# remover duplicados
df = df.drop_duplicates(subset=["codigo"])

df.head()

,codigo,preco,area_m2,quartos,banheiros,vagas,texto_completo
0,LOC1001,"R$ 3.200,00",35,2,1,0,Sobrado aceita pet mobiliado Código: LOC1001 R...
1,LOC1002,"R$ 2.900,00",95,0,2,0,"Casa mobiliado Código: LOC1002 R$ 2.900,00 Cas..."
2,LOC1003,"R$ 0,00",110,2,1,0,Sobrado Código: LOC1003 Sobrado em rua tranqui...
3,LOC1004,"R$ 1.400,00",72,1,2,0,Apartamento aceita pet mobiliado Código: LOC10...
4,LOC1005,"R$ 2.900,00",68,1,2,1,"Studio mobiliado Código: LOC1005 R$ 2.900,00 S..."


In [ ]:
df.to_csv("dados_imoveis.csv", index=False)

print("Arquivo salvo!")

Arquivo salvo!


# Relatório de ETL - LocaFácil

Aluno: Renato Parra Silva
Disciplina: Ciência de Dados e Aprendizagem de Máquina

## 1. Objetivo

Realizar o processo de ETL (Extract, Transform, Load) a partir do site fictício LocaFácil, extraindo os anúncios de imóveis, tratando inconsistências e organizando os dados em uma tabela para análise.

## 2. Fonte de dados

Os dados foram extraídos do site LocaFácil, disponível em:
https://computacao-aplicada.github.io/scraping/

O site foi desenvolvido para fins de estudo e contém dados fictícios de anúncios imobiliários.

## 3. Extração

Nesta etapa, foram acessadas as páginas do site para coletar os anúncios disponíveis.  
A extração foi feita com Python, utilizando `requests` para acessar o HTML e `BeautifulSoup` para localizar os elementos da página.

## 4. Transformação

Após a extração, os dados passaram por tratamento para melhorar a qualidade da base.  
Foram realizadas operações como:
- remoção de valores ausentes em campos importantes;
- remoção de registros duplicados;
- padronização de colunas;
- organização dos dados em um DataFrame.

## 5. Carga

Após o tratamento, os dados foram armazenados em um DataFrame do pandas e exportados para um arquivo CSV, permitindo reutilização posterior.

## 6. Resultados

Foi possível extrair os anúncios das páginas do LocaFácil e organizá-los em formato tabular.  
A base final contém informações como código do imóvel, preço, área, quartos, banheiros e vagas.

Durante o processo, foram observadas inconsistências intencionais no site, como dados faltantes e possíveis duplicidades, o que exigiu tratamento na etapa de transformação.

## 7. Conclusão

O processo de ETL permitiu transformar dados semiestruturados de páginas HTML em uma base organizada para análise.  
A atividade mostrou a importância das etapas de extração, limpeza e armazenamento dos dados, além de evidenciar problemas comuns em bases reais, como valores ausentes, duplicidades e falta de padronização.
